# Kalman Filter Price-Target Model — Fused Coregionalised Panel

**Notebook form of `pymc_kalman_filter_pt.py`**, aligned with
`probabilistic_ml_model/pymc_models/KalmanFilterModel.py`.

The cross-sectional spine is the **fused coregionalised (rank-1 ICM) panel model**
(`build_fused_kalman_pt_model`):

- **Model B spine** — a rank-1 Intrinsic Coregionalization Model (ICM) over the
  `(isin, time, y_series)` response tensor: the response series share the latent per-ISIN
  factor `mu_isin` via per-series loadings (primary anchored at 1), with direct per-series
  intercept/slope on the collapsed cross-section (a zero-anchored random-walk deviation is
  re-added only for genuine `T > 1` panels) and a per-series noise diagonal `sigma_series`.
- **Model A refinement** — the risk-aware `expected_return → risk_adj_return`
  latent (with a non-centred logit-normal `achieve_prob`) *is* the GRW baseline
  `mu_isin`, and the heteroscedastic scale `sigma_isin = sigma_base · (1 + cv) / √n`
  replaces the cv-free form. The risk adjustment is keyed on **systematic risk**
  (`feat_avg_beta`, the NULL-aware mean of `beta_{1y,2y,5y}`) rather than analyst
  conviction — replacing the earlier expected-volatility (`feat_vol_*`) penalty. Both the
  beta penalty and a second **size tilt** `− size_loading · z(feat_mcap_vs_3yavg)` enter
  **additively** as learned, sign-fixed loadings on `risk_adj_return`, demoting names re-rated
  above their own 3-year-average market cap (a mean-reversion tilt).

The workflow has two halves:

| Sections    | Scope                                                                                                                        |
|-------------|------------------------------------------------------------------------------------------------------------------------------|
| **§1–§10**  | Fused cross-sectional panel — one row per ISIN from `pml.mv_pymc_kalman_pt`                                                  |
| **§11–§14** | Single-security / cohort time-series — the `KalmanFilterPriceTarget` GRW filter on the embedded `*_ago` price-target history |

**Schema is the single source of truth (`pml` schema):** MV `pml.mv_pymc_kalman_pt`,
catalogue `pml.vw_pymc_feature_catalogue WHERE model_target = 'kalman_pt'`, coords
`pml.vw_pml_df_coords`.

> **Design note.** This notebook *orchestrates* the functions defined in
> `pymc_kalman_filter_pt.py`; it does not re-implement them. The module stays the
> single source of truth, and each section here calls into it and keeps the returned
> artifact at the top level so you can inspect it interactively.

## §0 — Environment & imports

`PYTENSOR_FLAGS` must be set **before** PyTensor/PyMC are first imported. On Windows
the flag parser strips backslashes from the `cxx` path (`C:\msys64\...` →
`C:msys64...`, a nonexistent compiler that hangs the *"Compiling new CVM"* step), so we
emit the g++ path with **forward slashes**, which survive the parser. If you launched
this kernel after `. .\set_env.ps1`, the existing flag is respected; otherwise we
auto-detect g++ and set a safe value here.

In [ ]:
import pytensor

# ── PyTensor backend guard — MUST run before the first ``pymc``/``pytensor`` import.
# Normalises PYTENSOR_FLAGS to the pure-Python / numba VM (``cxx=``), stripping any
# inherited ``cxx=<g++ path>``: nutpie compiles the logp via its own numba path and
# does not need PyTensor's C linker.
#
# Root cause of the historical empty-stderr ``CompileError (return status=1)``
# (diagnosed 2026-07-10): the actual compiler ``cc1plus.exe`` resolves its DLLs via
# PATH from ``C:\msys64\ucrt64\bin``; in an IDE-managed Jupyter kernel that dir is
# usually NOT on PATH, so cc1plus dies with STATUS_DLL_NOT_FOUND and zero output.
# ``g++.exe`` existing on disk is NOT sufficient — the previous version of this cell
# probed exactly that and re-armed a broken toolchain. Opt back into the C backend
# with PML_ENABLE_PYTENSOR_C=1 set BEFORE the kernel starts (set_env.ps1 probes an
# actual compile and prepends ucrt64\bin to PATH first).
from probabilistic_ml_model import _pytensor_env  # noqa: F401 — runs force_python_vm()

_cxx = pytensor.config.cxx
print("PyTensor", pytensor.__version__, "| cxx =", repr(_cxx),
      "(pure-Python/numba VM)" if not _cxx else "(C backend — opt-in)")


In [ ]:
import logging
import os
import pymc as pm
import numpy as np
import arviz_plots as azp
from xarray import DataTree
from sqlalchemy import create_engine

# Section functions live in the module (single source of truth) — import, don't re-define.
import pymc_kalman_filter_pt as kf
from pymc_kalman_filter_pt import (
    RANDOM_SEED,
    setup_plotting, resolve_db_url,
    load_kalman_df, load_feature_catalogue, resolve_feature_roles,
    run_eda, map_state_space_features, prepare_kalman_panel_inputs,
    build_panel_model, run_prior_predictive, sample_posterior, run_posterior_predictive,
    run_diagnostics, summarize_panel_screen, compute_cvar_aware_book, export_analytics,
    run_single_isin_filter, run_single_isin_stochastic_vol,
    run_mingled_cohort_filter, run_mingled_cohort_stochastic_vol,
    run_granular_forest, run_granular_further_views,
    run_summary, run_recommendations,
    present_group_effects,
)

logging.basicConfig(level=os.environ.get("LOG_LEVEL", "INFO"))
setup_plotting()
engine = create_engine(resolve_db_url())
print("Setup complete — RANDOM_SEED =", RANDOM_SEED, "| orchestrating:", kf.__file__)

## §1 — Data load & feature-role resolution

- `load_kalman_df` — cross-sectional `pml.mv_pymc_kalman_pt` snapshot (one row per ISIN,
  filtered to `observed_pt IS NOT NULL` and next earnings in the modelling horizon).
- `load_feature_catalogue` — the `kalman_pt` rows of `pml.vw_pymc_feature_catalogue`.
- `resolve_feature_roles` — groups columns by `pymc_role` (catalogue SSOT, MV-schema
  fallback): predictors, coords, responses, classification coords, fiscal-calendar and
  day-count columns.

In [ ]:
kalman_df = load_kalman_df(engine)
feature_catalogue = load_feature_catalogue(engine)
roles = resolve_feature_roles(kalman_df, feature_catalogue)
kalman_df.head()

## §2 — Exploratory data analysis

`run_eda` renders the EDA panels through a state-space lens: drift features → the
state-transition mean (`beta` slopes), noise wideners → the measurement-noise scale
`sigma_obs`. Includes the missingness overview, implied-upside ridges by industry,
coord cardinality, the winsorised `feat_*` distributional summary, the observation-noise
widener marginals, and the per-group implied-upside forests (an EDA preview of the §5
group effects).

**Interactive panels (Plotly).** Three panels render as interactive Plotly figures, each
with a matplotlib / seaborn fallback when Plotly is unavailable:

- **Implied upside vs fused-panel drivers** — a hoverable faceted scatter over the
  drivers `build_fused_kalman_pt_model` actually consumes: the systematic-risk
  `feat_avg_beta` penalty, the `feat_mcap_vs_3yavg` size discount, the new short-horizon
  momentum `feat_one_day_return`, and the `sigma_obs` wideners (consensus-noise CV, 6m
  volatility). Hover surfaces ticker / name / sector so individual names are
  identifiable. This replaces the earlier static upside-vs-noise/vol scatter, which
  predated the model's switch from the expected-volatility penalty to the beta + size
  tilts.
- **`feat_*` collinearity heatmap** — an interactive Spearman-ρ heatmap with cell labels
  and hover.
- **Momentum term-structure scan** — a new statistical summary: the Spearman ρ of each
  drift feature against implied upside, ρ-sorted, with `feat_one_day_return` highlighted
  so its highest-frequency momentum signal can be read against the 1m/3m/6m/1y → YTD → multi-year
  horizons (table always; interactive ρ-bar when Plotly is present).

> Skip this cell for a faster run — nothing downstream depends on it.

In [ ]:
run_eda(kalman_df, roles)

## §3 — State-space feature mapping

`map_state_space_features` maps the `feat_*` columns onto Kalman roles and returns the
drift-feature list (state-transition mean / `beta` slopes). Alongside the analyst-trail
drifts and `feat_total_return_ytd`, the drift block now also carries a short-horizon
momentum signal `feat_one_day_return` (last day's price change, sourced from `one_day_pct`)
and a **curated momentum ladder** of realised-return features — the rolling
`feat_total_return_1m`, `feat_total_return_3m`, `feat_total_return_6m`,
`feat_total_return_1y`, the long-horizon `feat_total_return_5y` / `feat_total_return_10y`,
and the `feat_tr_cagr_3y` / `feat_tr_cagr_5y` compound-growth rates (sourced from the
`total_return` / `tot_return_pct_cagr_*` columns) — plus `feat_mv_ev_drift` — the drift of the `market_cap / enterprise_value` ratio (equity share of EV) across the fiscal-year trail, an equity re-rating / de-leveraging signal sourced from the matched `market_cap` / `enterprise_value` lag pairs in `mv_pymc_kalman_pt`. All are registered as `kalman_pt`
`mutable_predictor`s in `pml.vw_pymc_feature_catalogue`.

> **Available vs. consumed.** `mv_pymc_kalman_pt` also emits the rest of the
> realised-return family — `feat_total_return_{1d,5d,1w,3y,mtd,qtd}`, the calendar-year
> buckets `feat_total_return_{2021…2025}`, and `feat_tr_cagr_{1y,10y}` — so they are
> **available** in the MV / catalogue but are intentionally **kept out of the drift
> matrix**: the sub-monthly windows are microstructure noise, the period-to-date returns
> overlap the rolling windows, and the calendar-year buckets are stale and mutually
> collinear, so they add little independent signal to the state-transition mean.

Together the retained members span the momentum term structure —
one day → 1m/3m/6m/1y → YTD → multi-year: `feat_one_day_return` contributes the
highest-frequency reversal/continuation signal, while the long-horizon members are mutually
collinear, so the weakly-informative `beta` prior regularises their individual slopes.

> **Leakage guardrail:** `feat_implied_upside = (observed_pt − last_price)/last_price` is
> a deterministic function of the response, so it must never enter the drift-predictor
> matrix. The mapping asserts this.

> **Cross-cutting size/trend feats.** `mv_pymc_kalman_pt` also emits `feat_mcap_trend_1y`,
> `feat_mcap_vs_3yavg` and `feat_ev_vs_3yavg` (market-cap / EV size-and-trend signals
> shared across every `mv_pymc_*` view). These are **not** drift predictors:
> `feat_mcap_vs_3yavg` (`= market_cap / market_cap_3yavg`) is consumed separately in §5 as
> an additive **size tilt** (`− size_loading · z(feat_mcap_vs_3yavg)`) on `risk_adj_return`, while `feat_mcap_trend_1y` and
> `feat_ev_vs_3yavg` are carried as provenance only.

In [ ]:
drift_features, mapping = map_state_space_features(kalman_df)
mapping

## §4 — Fused-panel data containers

`prepare_kalman_panel_inputs` filters to log-space-usable rows and builds the
`KalmanPanelInputs`: the standardised `(isin, time, y_series)` response tensor `Y`, the
fiscal-anchor time matrix `t_scaled`, the standardised drift design matrix (now including
the short-horizon `feat_one_day_return` momentum predictor alongside the analyst-trail and
1m/3m/6m/1y → YTD → multi-year drifts), the volatility / dispersion-cv / √n noise drivers, the systematic-risk (`feat_avg_beta`) and
size (`feat_mcap_vs_3yavg`) drivers, and the categorical group-effect coords. The latter
two are carried raw and z-scored inside `build_fused_kalman_pt_model`, where they drive the
additive beta and size tilts (`− risk_loading · z(avg_beta) − size_loading · z(mcap_vs_3yavg)`) on `risk_adj_return` (§5).

> **Primary response = log uplift.** The first (anchor) response series is
> `feat_log_uplift = log1p(feat_implied_upside)`, winsorised to implied upside
> ∈ [−95 %, +500 %] before the log. Modelling the **log** uplift keeps the
> reconstructed price target strictly positive
> (`expected_pt = last_price · exp(log_uplift)`): the raw decimal `feat_implied_upside`
> is unbounded below, so de-standardising the Gaussian baseline linearly
> (`last_price · (1 + eu)`) produced **negative** `expected_pt` for names trading far
> above their analyst targets (extreme YTD momentum run-ups). `observed_pt` is dropped
> as a response — it is a near-collinear price-level restatement of the primary series.

> **Response coverage guard (`KALMAN_RESPONSE_COVERAGE_MIN`, default 0.60).** A
> NON-PRIMARY response series that is mostly `NULL` (e.g. `feat_pt_drift` when the
> `price_target_*_ago` trail is unpopulated) standardises to a near-constant (mostly-zero)
> column whose rank-1 ICM loading is **unidentified** — the flat `loading × mu_isin` ridge
> that previously froze the sampler (R-hat 4.45 / ESS 4.3). The guard measures each
> non-primary series' finite-coverage fraction on the **raw** column (before the model's
> nan→0 fill) and its standardised variance, and drops any below the coverage gate or with
> ~0 variance, printing a `[guard] dropping degenerate response series …` line (no silent
> truncation). On the current snapshot this drops `feat_pt_drift` → `D=1`, and the ICM
> degenerates to a clean cross-section; a future well-populated second series reactivates it
> automatically. Standardisation also switched to **NaN-aware** `nanmean`/`nanstd` so a
> single `NaN` can no longer zero an entire kept series. Watch the printed `Y shape` /
> `response_names` below for the surviving series.

In [ ]:
panel = prepare_kalman_panel_inputs(kalman_df, roles, drift_features)
print("Y shape (isin, time, y_series):", panel.Y.shape)
print("response_names:", panel.response_names)
print("drift_names   :", panel.drift_names)

## §5 — Build the fused coregionalised model

`build_panel_model` wraps `build_fused_kalman_pt_model` and renders the model graph.

### Generative form

Per ISIN $i$, fiscal-anchor time $t$, and response series $d$ (the primary anchor series
$d=0$ is `feat_log_uplift`). The fused model collapses two notebook models into one
builder: a systematic-risk-conditioned cross-sectional drift baseline (**Model A**) supplies
the per-ISIN level of a rank-1 coregionalised (ICM) cross-sectional spine (**Model B**).

**Model A — systematic-risk-conditioned drift baseline.** A hierarchical regression on the
standardised drift design $X^{\text{drift}}$ with crossed, sum-to-zero group intercepts
(`trading_region` /`region` / `sector` / `size_class` / `style_class`) at a **fixed** scale (the group SD is
*not* learned):

$$\eta_i = X^{\text{drift}}_i\,\beta + \sum_g \big(e^{(g)}\big)_{[i]}, \qquad \beta \sim \mathcal{N}(0, 1),\ \ e^{(g)} \sim \text{ZeroSumNormal}(\sigma_{\text{group}}{=}0.25)\ \text{(fixed)}$$
$$\text{expected\_return}_i = \eta_i \qquad\text{(deterministic structural mean — no per-ISIN signal latent)}$$
$$\text{risk\_adj\_return}_i = \text{expected\_return}_i - \lambda\, z(\text{avg\_beta})_i - \lambda_{\text{size}}\, z(\text{mcap\_vs\_3yavg})_i \;=:\; \mu^{\text{isin}}_i$$

The risk loading $\lambda$ (a **learned**, sign-fixed `HalfNormal` whose prior scale is
`risk_penalty`, default `0.1`) tilts the return **additively** by the **standardised**
(z-scored) average market **beta** — the CAPM systematic-risk measure — a monotone
$-\lambda\,z$ cross-sectional discount that demotes high-beta names and lifts low-beta ones. Beta isolates the priced,
non-diversifiable risk that should discount expected return, replacing the earlier
expected-volatility penalty (which conflated idiosyncratic noise and, fed as a raw
percent level ≈ 45, produced an $\exp(-4.5)\approx0.01$ discount that annihilated the
regression). Expected volatility (`feat_vol_*`) is retained as a provenance container
only.

A second additive **size tilt** $-\lambda_{\text{size}}\, z(\text{mcap\_vs\_3yavg})_i$
(learned loading, prior scale `size_penalty`, default `0.1`) rides alongside the beta
penalty. `feat_mcap_vs_3yavg = market_cap / market_cap_3yavg` is each firm's current size
relative to its own 3-year average, so with $\lambda_{\text{size}} > 0$ names re-rated
**above** their history are discounted and names **below** earn a premium — a
mean-reversion tilt structurally identical to the beta discount. Set `size_penalty=0` to
disable it; the per-ISIN tilts are exposed as the `risk_tilt` / `size_tilt` Deterministics
for attribution. A non-centred logit-normal
$\text{achieve\_prob}_i = \sigma(\mu_{\text{logit}} + \sigma_{\text{logit}}\,z_i)$ is
carried alongside for downstream consumers.

**Model B — rank-1 coregionalised cross-section (ICM).** The $D$ response series
(`feat_log_uplift`, and `feat_pt_drift` only when it clears the §4 coverage guard) share
the single latent per-ISIN factor $\mu^{\text{isin}}$ through per-series loadings $\ell_d$ —
a **rank-1 Intrinsic Coregionalization Model** (the PyMC *MOGP-Coregion-Hadamard* pattern),
the primary series anchored at $\ell_0 \equiv 1$ (fixing the factor scale), others
$\sim \text{LogNormal}(0, 0.3)$ (median 1, strictly positive — **sign-fixed** so the loading
cannot flip the latent factor's sign, removing the multimodal $\ell_d\,\mu^{\text{isin}}$
ridge). Each series gets a **direct** intercept $\alpha_d$ and time-slope $\beta_d$ (NOT a
random walk):

$$\text{reg}_{i,t,d} = \alpha_d + \beta_d\,t^{\text{scaled}}_{i,t} + \ell_d\,\mu^{\text{isin}}_i, \qquad \alpha_d \sim \mathcal{N}(0,1),\ \ \beta_d \sim \mathcal{N}(0,0.5)$$

> **Why direct intercepts, not a random walk (the ESS ≈ 7 fix).** The previous build wrote
> the level as a single-step walk $\alpha_{0,d} = \sigma^{\alpha}_d\,z^{\alpha}_{0,d}$ — a
> product of **two** scalars for **one** intercept. On the collapsed $T = 1$ slice the
> likelihood sees only the product, so $(\sigma^{\alpha}_d, z^{\alpha}_{0,d})$ is
> non-identified: a ridge (not a funnel — it survives with **0 divergences**) that froze
> the sampler at $\text{ESS} \approx 7$, $\hat R \approx 1.5$, with $\sigma^{\alpha}_d$
> within-chain variance $\to 0$ (the arviz $\hat R$ divide-by-zero) and $\sigma_{\text{group}}$
> squeezed to $\approx 0$. Direct intercepts remove the redundant scale entirely — the
> model now mixes ($\hat R \approx 1.0$, ESS in the hundreds–thousands). A genuine time
> panel ($T > 1$, `collapse_time=False`) re-adds a **zero-anchored** GRW *deviation* on top
> of $\alpha_d$ / $\beta_d$, where $\sigma^{\alpha}_d, \sigma^{\beta}_d \sim \text{HalfNormal}(0.5)$
> are identified by the multiple time steps.

> **Why the loading is sign-fixed + the zero-variance guard (the R-hat 4.45 / ESS 4.3
> fix).** A free $\ell_d \sim \mathcal{N}(1, 0.5)$ let $\ell_d\,\mu^{\text{isin}}$ flip sign
> (factor and loading are only jointly identified up to a shared sign) — bimodal and
> non-mixing. Worse, when the **second response series was degenerate** (`feat_pt_drift`
> standardised to ≈0 because the `price_target_*_ago` trail is mostly `NULL`), its loading
> had no data to pin it and wandered (sd ≈ 1.5) while its $\alpha_d$/$\beta_d$ pinned at 0 —
> a flat ridge that collapsed the global step size and froze every chain. The fix is the §4
> **coverage guard** (drops such series upstream) plus the **`LogNormal` sign-fixed
> loading** above and a **zero-variance assertion** in the builder, so a degenerate series
> can never enter the ICM.

**Heteroscedastic robust likelihood.**

$$\sigma^{\text{isin}}_i = \sigma_{\text{base}}\,\frac{1 + \text{cv}_i}{w_i}, \qquad \sigma_{\text{base}} \sim \text{Exponential}(1),\ \ w_i = \sqrt{n^{\text{analysts}}_i / \operatorname{median}(n)}$$
$$Y_{i,t,d} \sim \text{StudentT}\!\big(\nu,\ \text{reg}_{i,t,d},\ \tau_d\,\sigma^{\text{isin}}_i\big), \qquad \tau_{d>0} \sim \text{HalfNormal}(0.25),\ \ \nu = 2.5 + \text{Gamma}(2, 0.1)$$

The consensus-noise coefficient of variation $\text{cv}_i$ widens noisy names; the
**median-normalised** analyst-precision weight $w_i$ tightens high-coverage names while
keeping the median name at $w_i \approx 1$ (raw $\sqrt{n}$ drove $\sigma^{\text{isin}}$
~10× too small on the z-scored response, collapsing the screen to full pooling). The hard
floor $\nu \ge 2.5$ guarantees finite variance and removes the improper
$\sigma_{\text{base}}\to0,\ \nu\to0$ corner. The posterior `expected_pt` de-standardises
the primary log-uplift series back through `last_price · exp(log_uplift)` — strictly
positive by construction. Pass `robust=False` for the Normal-likelihood twin (default is
the Student-t panel likelihood, which absorbs analyst outliers).

Each output additionally carries its own noise scale $\tau_d$ (the ICM $\kappa$ diagonal;
primary anchored at 1, others $\sim \text{HalfNormal}(0.25)$) so a noisier secondary
series does not inflate the primary series' $\sigma$ — the heteroskedastic-noise pattern of
the PyMC *GP-Heteroskedastic* example, here positive-by-construction rather than via a
log-link.

### NUTS-inplace-safety constraints baked into the builder

- `√n_analysts` is precomputed in NumPy and passed as `pm.Data` (never `pt.sqrt` on an
  integer container).
- `sigma_base` uses `pm.Exponential` (not `HalfNormal`) so no `Abs→Sqrt` rewrite fuses
  into an inplace `Composite` op.
- The GRW uses an explicit **diagonal** innovation parameterisation instead of
  `LKJCholeskyCov` (whose onion-method Beta→Normal chain triggers an inplace rewrite
  NUTS rejects).

### Sampler-geometry parameterisation

- `mu_global` is **dropped** — `mu_isin` is the model's sole identified level, removing
  the flat `mu_global` ↔ walk-level ridge that drove the ~0.001 step-size /
  max-tree-depth regime.
- Crossed group intercepts use a **fixed-scale `ZeroSumNormal`** (`GROUP_EFFECT_SCALE =
  0.25`): the sum-to-zero constraint removes the additive level ridge, and **fixing** the
  group SD removes the variance-partition ridge entirely. A *learned* `sigma_group` is
  structurally non-identified on the collapsed `T = 1` slice — one cross-section cannot
  separate between-group dispersion from the residual base scale — so both
  parameterisations stalled (non-centred stuck at R-hat ≈ 1.52 / ESS ≈ 7, collapsed to ≈
  0.004; centred at R-hat ≈ 4.5 / ESS ≈ 4, wandering ≈ 0.22). The group *effect* vectors
  themselves already mix well (ESS ≈ 4k–9k); only the learned scale was stuck, so it is
  fixed (Gelman: fix the group SD when one slice can't identify it). `industry` is excluded
  (near-nested under `sector`); per-coord `sigma_<col>` are re-exposed as the **empirical
  between-group sd** of each effect — a genuine, well-identified per-coord number, no
  longer four aliases of one stuck scalar.
- `expected_return` is the **deterministic** structural mean `mu_reg`; the old per-ISIN
  signal latent (with its own `sigma_expected_return`) is removed and idiosyncratic
  per-ISIN dispersion is carried by the Student-t `sigma_isin` instead. The beta and size
  penalties are **additive, sign-fixed learned loadings** (`− risk_loading · z(β)`,
  `− size_loading · z(size)`; prior scales `risk_penalty` / `size_penalty`). The earlier
  **multiplicative** `exp(−penalty · z)` factor only rescaled the magnitude of the
  zero-mean `expected_return` deviation and **inverted** the discount for below-average
  names; combined with the earlier stuck learned group scale the cross-section collapsed toward
  full pooling — the **stale, uniformly
  non-negative** `expected_upside` / `log_uplift`. The additive tilts are monotone for
  either sign and add identified mean structure.
- The rank-1 ICM **coregion loading is sign-fixed** (`mu_isin_loading ~ LogNormal`) and a
  **zero-variance / coverage guard** (§4 `KALMAN_RESPONSE_COVERAGE_MIN` + a builder
  assertion) keeps any degenerate response series out of the ICM — the final fix that
  unfroze the R-hat 4.45 / ESS 4.3 run.

In [ ]:
robust = True
model = build_panel_model(panel, robust=robust)
pm.model_to_graphviz(model)

## §6 — Prior predictive checks

`run_prior_predictive` draws 1000 prior samples of `expected_return`, `risk_adj_return`,
`achieve_prob` and `sigma_isin`, then renders three panels so the §5 refinements are
visible **before any data is seen**:

1. **Prior implied upside vs empirical** — the fused baseline `risk_adj_return` is
   de-standardised onto the primary `feat_implied_upside` series (via
   `panel_posterior_upside`) and overlaid on the empirical `observed_pt / last_price − 1`.
   A sane prior should bracket the empirical upside without concentrating mass at
   implausible returns.
2. **`achieve_prob` (logit-normal)** — the Model-A achievement probability
   $\sigma(\mu_{\text{logit}} + \sigma_{\text{logit}}\,z_i)$, which should span $(0,1)$
   without piling up at the bounds.
3. **`sigma_isin` (heteroscedastic scale)** — the measurement scale
   $\sigma_{\text{base}}(1+\text{cv})/w_i$, a positive right-skewed prior.

The predictive draw forces the pure-Python linker via `compile_kwargs`
(`get_pytensor_compile_kwargs()`), so it cannot stall on C compilation.

In [ ]:
prior_idata = run_prior_predictive(model, panel)

## §7 — Posterior inference (NUTS)

`sample_posterior` tries `nutpie → numpyro → pymc` in priority order (whichever is
installed; pure-Python `pymc` NUTS is always the fallback) and merges the prior groups
into the posterior `DataTree` for one-object downstream access.

**Sampler settings** (see `sample_posterior`): `draws=1000, tune=1000, chains=4,
cores=4, target_accept=0.9`. 4×1000 = 4000 draws and `cores=4` runs all chains in
parallel. `target_accept` was **relaxed 0.97 → 0.9**: the high value was a band-aid that,
against the degenerate-series ICM ridge below, drove the step size toward 0 and froze
every chain; with that ridge removed the posterior is well-conditioned and the default-ish
0.9 mixes fast.

> **The decisive fix is structural, not budgetary.** An earlier build froze completely —
> **max R-hat 4.45, min ESS 4.3, 0 divergences**, with nearly every parameter degenerate.
> The trigger was the **second response series `feat_pt_drift`**: it is `NULL` whenever the
> lagged `price_target_*_ago` trail is unpopulated (`pml.target_drift` returns `NULL` when
> every consecutive pair has a `NULL`/0 predecessor), so after z-scoring it standardised to
> a near-constant column. Its rank-1 ICM loading `mu_isin_loading[feat_pt_drift]` was then
> **unidentified** (it wandered with sd ≈ 1.5 while `alpha`/`beta_slope` for that series
> pinned at 0) — a flat `loading × mu_isin` ridge that, under `target_accept=0.97`,
> collapsed the global NUTS step size and froze the whole posterior.
>
> Three changes remove it: (1) a **coverage guard** in `prepare_kalman_panel_inputs` drops
> response series with low finite-coverage / ~0 standardised variance (so `feat_pt_drift`
> is dropped on the current snapshot → the ICM collapses to a clean `D=1` cross-section);
> (2) `build_fused_kalman_pt_model` **sign-fixes** the coregion loading
> (`mu_isin_loading ~ LogNormal`, no sign-flip multimodality) and **asserts** no
> zero-variance series enters the ICM; (3) the MV emits **min-points-guarded, winsorised**
> drift columns plus `*_n` valid-pair counts so a sparse trail is gatable at source. These
> compose with the earlier identifiability fixes (collapsed `T=1` slice, deterministic
> `expected_return`, fixed-scale `ZeroSumNormal` group effects) so `draws=1000` /
> `tune=1000` / `target_accept=0.9` clear **R-hat < 1.01** and **ESS > 400** comfortably.

> This is the long-running cell. With the `cxx` path fixed (§0), nutpie compiles via
> numba — no g++ stall.

In [ ]:
# cores=1: run chains sequentially inside the IDE-managed Jupyter kernel. Launching
# nutpie's parallel native worker threads (cores=4) inside the embedded kernel crashes
# the kernel process on Windows ("Connection to IDE-Managed Server is lost" — an
# uncatchable native crash, no traceback). The standalone script path keeps cores=4.
idata = sample_posterior(model, prior_idata, cores=1)
print("Group effects fitted:", present_group_effects(idata))
idata.posterior

## §8 — Posterior predictive checks

The fused likelihood `target_pct_obs` is the standardised `(isin, time, y_series)`
response tensor, so `run_posterior_predictive` pools the replicated draws against the
observed standardised responses and reports calibration four ways:

- **(a) ArviZ `plot_ppc_dist`** (ECDF kind) — replicated vs observed distribution.
- **(b) Pooled ECDF overlay** — ~60 posterior-predictive draws (light) against the
  observed ECDF (bold), a robust fallback for the multidimensional response.
- **(c) Per-`y_series` 94 % coverage** — the fraction of observed values inside the
  per-series $[\,3\%,\,97\%\,]$ posterior-predictive quantile band, which should land
  near **0.94**. Material under-coverage flags an over-confident `sigma_isin`;
  over-coverage flags an inflated one.
- **(d) ArviZ `plot_ppc_pit`** — the PIT calibration plot should track the diagonal
  (best-effort; skipped cleanly if the multidim PIT is unavailable).

In [ ]:
run_posterior_predictive(model, idata, panel)

## §9 — MCMC diagnostics

`run_diagnostics` reports R-hat / bulk-&-tail-ESS summaries, the divergence count, and
trace / rank-dist / forest views over the fused-model hyper-parameters: the global
scalars (`FUSED_SCALAR_VARS`), the per-coord between-group sds `sigma_<coord>`, the
drift slopes `beta`, and the GRW innovation scales `sigma_alpha_innov` /
`sigma_beta_innov`. Empty-dim variables (absent group effects) are skipped and reported.

The convergence gates follow Vehtari et al. (2021): **R-hat < 1.01** and **ESS > 400**.
The earlier run froze entirely here — **max R-hat 4.45, min ESS 4.3** with nearly every
parameter degenerate — because the degenerate `feat_pt_drift` response left its ICM loading
unidentified (see §4 / §7). With that series dropped by the coverage guard, the loading
sign-fixed, and the builder's zero-variance assertion in place (on top of the §5
deterministic-`expected_return` / fixed-scale group-effect reparameterisation), the chains
now mix: this report should show **R-hat < 1.01**, **ESS > 400**, **0 divergences**, and an
empty degenerate-var list. If a future snapshot repopulates `feat_pt_drift` (≥ 60 %
coverage), it re-enters as `D = 2` and the per-series ICM diagnostics
(`mu_isin_loading`, `sigma_series`) reappear — they should clear the same gates.

In [ ]:
run_diagnostics(idata, panel)

## §10 — Expected price targets: posterior screen & export

- `summarize_panel_screen` → a `ScreenContext` carrying the posterior `expected_upside`
  (`eu`) and `expected_pt` (`ept`) draws over `(chain, draw, isin)`, the per-ISIN
  screening `results` table (sorted by expected upside), and the structural-TS
  Monte-Carlo `mc_summary`.
- `expected_pt` is reconstructed multiplicatively from the de-standardised log-uplift
  baseline (`last_price · exp(log_uplift)`), so it — and its 94 % HDI band — is positive
  by construction.
- The `mc_summary` `er_*` columns are genuine **decimal returns**: the latent
  `risk_adj_return` / `sigma_isin` are de-standardised onto the log-uplift scale before
  the simulation, then `expm1`-mapped, so `prob_pos = P(return > 0)` (not P(beats the
  cross-sectional average upside)).
- The per-ISIN `results` table also surfaces a representative subset of the
  realised-return drift inputs as reporting columns — `total_return_ytd_pct`,
  `total_return_5y_pct`, `total_return_10y_pct`, `tr_cagr_3y_pct` — next to the posterior
  `expected_upside_pct` so the smoothed expectation can be read against realised momentum.
  The rest of the §3 curated momentum ladder (the rolling 1m/3m/6m/1y returns and
  `feat_tr_cagr_5y`) feeds the drift matrix but is not echoed back into the screen.

`export_analytics` de-standardises the screen and (when `write=True`) appends one row per
ISIN to `analytics.kalman_filtered_price_targets`, mapping the fused coregionalised +
systematic-risk-conditioned posterior onto the table's Kalman columns:

| analytics column                                   | source (fused posterior)                                            |
|----------------------------------------------------|---------------------------------------------------------------------|
| `price_target_kalman` / `kalman_estimate`          | posterior-mean `expected_pt` (de-standardised smoothed target)      |
| `kalman_variance`                                  | posterior variance of `expected_pt`                                 |
| `implied_return_kalman` / `expected_upside_kalman` | posterior-mean `expected_upside` (decimal)                          |
| `kalman_gain`                                      | logit-normal `achieve_prob` (smoother confidence analogue)          |
| `signal_strength`                                  | `\|E[risk_adj_return]\| / sd(risk_adj_return)` (conviction z-score) |
| `cvar_book_weight`                                 | normalised long-book weight (held names sum to 1 / 100 % gross)     |
| `cvar_5pct_kalman`                                 | per-name 5 % expected shortfall (CVaR) of the upside draws          |
| `reward_to_cvar`                                   | STARR ratio (expected upside / binding tail risk) used for sizing   |

The §10b CVaR sizing is computed once by `compute_cvar_aware_book` and shared with the
§14b recommendations, so the export and the screen agree exactly. Set `write=True` to
append via `export_to_analytics_db` (`DB_ANALYTICS_SCHEMA`, default `analytics`).

In [ ]:
screen = summarize_panel_screen(idata, panel)
results = screen.results
results.head(15)

### §10b — CVaR-aware risk analytics & sizing (RiskBook)

Single source of truth for the risk layer: per-name expected shortfall (CVaR of the
posterior upside draws), a reward-to-CVaR (STARR) ranking, and a per-name-capped long
book with a joint-draw portfolio expected shortfall. The resulting `RiskBook` feeds
**both** the §10c analytics export and the §14b recommendations, so the sized book is
identical across them (rather than being recomputed independently in each).

In [ ]:
# §10b CVaR-aware risk book (SSOT): per-name tail analytics + a STARR-ranked,
# per-name-capped long book. Reused by the §10c export and §14b recommendations below.
risk_book = compute_cvar_aware_book(idata, panel, screen, results)
risk_book.book.head(15)

In [ ]:
# Set write=True to persist to analytics.kalman_filtered_price_targets.
kalman_results = export_analytics(idata, panel, screen, risk_book=risk_book, write=True)
kalman_results.head()

## §11 — Single-ISIN time-series Kalman filter (+ §11b stochastic volatility)

§1–§10 are the **cross-sectional** panel: one row per ISIN, no real time axis. This
section switches to the **literal** single-security `GaussianRandomWalk` filter from
`KalmanFilterModel.py` to recover the quantity this model exists for — a **price target
evolving over time** with a credible band.

`run_single_isin_filter` reconstructs the time axis from the embedded `*_ago`
price-target cohort (`price_target_1w_ago`, `price_target_high_3m_ago`,
`price_target_median_1y_ago`, …), unpivoted via `build_price_target_history()` into a
`(isin, asof_date, price_target)` panel and **anchored on `income_statement_report_date`**
so the axis tracks the real reporting cadence. It picks the cohort name with the richest
history (most-recent earnings, largest market cap, ≥ 2 `*_ago` observations).

Because that cohort is short (~6–16 points) and **irregularly spaced** (1w, 1m, 3m, 6m,
1y) — exactly where an explicit latent random walk funnels — the fit uses the funnel-free
**marginalized** parameterization with a **structural trend** (`trend=True`). The latent
path is integrated out analytically into a single `MvNormal` likelihood whose covariance
carries **both** the random-walk (process) and observation (measurement) variances,

$$\Sigma_{st} = P_0 + \sigma_{\text{state}}^2\,\min(\tau_s, \tau_t) + \sigma_{\text{obs}}^2\,\delta_{st},$$

with $\tau$ the **real elapsed time** (years) between observations (`_resolve_time_deltas`).
The observed log-targets enter *only* as the data (never as the mean), so $\sigma_{\text{state}}$
and $\sigma_{\text{obs}}$ are genuinely identified; the smoothed path is recovered as the
analytic Kalman/RTS smoother mean $\mathbb{E}[x\mid y]$ and rendered with
`plot_price_target_path()`. The fit then projects forward to the next fiscal events via
`KalmanFilterPriceTarget.forecast()`.

`run_single_isin_stochastic_vol` re-fits the **same** series with the opt-in latent
log-volatility random walk + robust Student-t observations (§11b). The cell is guarded —
it degrades to an informative message if `DB_URL` is unset or no ISIN carries ≥ 2 `*_ago`
observations.

> **Shared driver.** Both this section and §12 route their fit + forecast through `fit_kalman_model` (in `pymc_kalman_filter_pt`), a thin wrapper over `KalmanFilterPriceTarget.fit` / `.forecast`. It adopts **spot anchoring** (`last_price` flows into the fit and the forecast) and resolves the forecast as-of anchor from the `last_updated` column plus the fiscal-calendar date coords.

In [ ]:
single_ctx = run_single_isin_filter(panel.frame, engine)
run_single_isin_stochastic_vol(single_ctx)

## §12 — Mingled-ISIN earnings-window cohort filter (+ §12b stochastic volatility)

Same state-space machinery as §11, but the time axis now represents an
**earnings-cohort consensus** rather than one security's history.
`run_mingled_cohort_filter` selects every ISIN whose `next_earnings` lands in the
**±10-day window** around today (`current_date ± INTERVAL '10 days'`, and on/after
2026-01-01), unpivots each name's `*_ago` cohort, and takes the cross-sectional
**median** price target at each shared `asof_date` — a single, time-ordered series of the
cohort consensus as it evolved across the earnings period.

The mingled series is again short and **irregularly spaced**, so it is fit with the same
funnel-free **marginalized** GRW (+ structural trend) whose `MvNormal` covariance carries
both the random-walk and observation variances scaled by real elapsed time. The three
quantities are compared on one axis: the posterior-mean Kalman-smoothed `expected_pt`
(with 94 % / 50 % HDI bands), the mingled cohort-median `observed_price_target`, and the
cohort-median `last_price` reference line.

`run_mingled_cohort_stochastic_vol` (§12b) re-fits with **stochastic volatility** — the
scalar `sigma_obs` replaced by a latent log-volatility random walk under a robust
Student-t likelihood, anchored on the **cohort-median** `feat_vol_{1m,3m,6m,1y}`
term-structure mapped onto the mingled `asof_date` grid by `build_realized_vol_path()`.
Both cells are guarded — they degrade cleanly when `DB_URL` is unset or the window yields
fewer than 2 mingled observations.

In [ ]:
mingled_ctx = run_mingled_cohort_filter(panel.frame, engine)
run_mingled_cohort_stochastic_vol(panel.frame, mingled_ctx)

## §13 — Granular earnings-cohort posterior-predictive forest (+ §13.1 further views)

Where §12 **mingles** the ±10-day earnings cohort into one median series and refits, this
section keeps the same cohort but goes the other way: it stays **per-ISIN granular** and
**reuses the already-fitted cross-sectional posterior** from §5–§10 (no refit).

`run_granular_forest` de-standardises each cohort ISIN's posterior-predictive log-uplift
back to price units (`last_price · exp(log_uplift)`), giving a posterior-predictive
distribution of the *expected price* (the simulated analyst target) per name. These are
rendered as an `arviz_plots` **posterior-predictive forest** with the realised analyst
targets overlaid as observation points, plus two reference layers that turn it into a
screen: a **reference HDI band** (the pooled `expected_pt` 94 % / 50 % posterior region —
names whose interval sits entirely outside the 94 % band are the cohort's relative
outliers) and a **cohort `last_price` line** so the targets read as implied upside.

`run_granular_further_views` (§13.1) adds two complementary views: (a) the same forest
keyed off the stored §10 `results` HDI columns (cohort-median `expected_pt_hdi_lo … hi`),
and (b) a per-chain KDE of the cohort-average expected upside with a 0 % break-even line
(the chain overlay doubles as a soft convergence check). When the cohort is large the
forest caps to the most extreme names by expected upside (top/bottom 20) to keep one row
per ISIN legible; the truncation is announced. Both cells are guarded against an unset
`DB_URL` or an empty cohort overlap.

In [ ]:
forest_ctx = run_granular_forest(idata, results, panel, screen, engine)
run_granular_further_views(prior_idata, panel, screen, forest_ctx)

## §14 — Comprehensive summary & actionable recommendations

`run_summary` consolidates the notebook into a decision-oriented read on the names
reporting within the **±10-day** earnings window, benchmarked against the broader
universe:

- **Cross-sectional** — earnings cohort vs the rest of the universe (names *not*
  reporting), on expected upside, share positive, credible-band width (uncertainty) and
  Kalman shrinkage vs raw consensus, read from the §10 `results` table.
- **Time-series** — the mingled cohort's reconstructed `*_ago` consensus trail (§12):
  oldest vs most-recent consensus, and the latest smoothed target's implied upside.
- **Distributional** — a posterior KDE of cohort-average vs universe-average expected
  upside with a 0 % break-even line.

`run_recommendations` (§14b) turns the posterior into risk-aware signals — read-only over
`idata`, the panel frame and `results`, reusing the §10 `compute_cvar_aware_book` so the
sizing matches the export exactly:

- **§1–§7 directional** — group- and name-level OVERWEIGHT / NEUTRAL / UNDERWEIGHT
  verdicts plus a band-width / coverage caution list.
- **§8 risk-adjusted** — reward per unit of *expected volatility* (the `feat_vol_*`
  term-structure; a separate portfolio-level risk budget, distinct from the panel's
  beta-based `risk_adj_return`), a forward Sharpe-like ranking that demotes
  high-upside / high-vol names.
- **§9 CVaR tails** — per-name expected shortfall (mean of the worst `ALPHA`-tail) of the
  upside draws, cross-checked against the Student-t Monte-Carlo 5 % return; with a low
  estimated `nu` these tails — not the means — drive sizing.
- **§10 CVaR-aware sizing** — a long book sized on the reward-to-CVaR (STARR) ratio with a
  per-name cap and a joint-draw portfolio expected shortfall, so no single fat-tailed name
  dominates the book's tail risk.

All blocks are guarded: the summary degrades to whatever upstream artifacts are present in
the kernel, so a partial read still prints when the DB-dependent sections were skipped.

In [ ]:
run_summary(results, screen, forest_ctx, mingled_ctx)
run_recommendations(idata, panel, results, screen, forest_ctx, risk_book=risk_book)

### §14.1 — Screen distribution & recommendation rankings (matplotlib)

In [ ]:
def plot_screen_overview(results_df, top_n=15):
    """

    Parameters
    ----------
    results_df
    top_n
    """
    eu = results_df["expected_upside_pct"].dropna() / 100.0
    fig, axes = plt.subplots(1, 2, figsize=(16, 6))

    # Expected-upside distribution
    axes[0].hist(eu, bins=60, color="#4C72B0", edgecolor="black", alpha=0.8)
    axes[0].axvline(eu.median(), color="#C44E52", ls="--", lw=2,
                    label=f"median = {eu.median():.1%}")
    axes[0].axvline(0.0, color="grey", ls=":", lw=1.5, label="break-even")
    axes[0].set_title("Cross-sectional expected-upside distribution")
    axes[0].set_xlabel("Expected upside")
    axes[0].set_ylabel("ISIN count")
    axes[0].legend()

    # Top-N ranked recommendations with HDI error bars
    top = results_df.sort_values("expected_upside_pct", ascending=False).head(top_n).iloc[::-1]
    lbl = top["ticker"].fillna(top["isin"].str[:6])
    lo = (top["expected_pt_hdi_lo"] / top["last_price"] - 1.0).to_numpy()
    hi = (top["expected_pt_hdi_hi"] / top["last_price"] - 1.0).to_numpy()
    mid = top["expected_upside_pct"].to_numpy() / 100.0
    xerr = np.vstack([np.clip(mid - lo, 0, None), np.clip(hi - mid, 0, None)])

    axes[1].errorbar(mid, range(len(top)), xerr=xerr, fmt="o",
                     color="#55A868", ecolor="#8C8C8C", capsize=3)
    axes[1].axvline(0.0, color="grey", ls=":", lw=1.5)
    axes[1].set_yticks(range(len(top)))
    axes[1].set_yticklabels(lbl)
    axes[1].set_title(f"Top {top_n} expected upside (94% HDI band)")
    axes[1].set_xlabel("Expected upside")

    plt.show()


plot_screen_overview(results)

### §14.2 — Interactive risk/return screen (plotly)

In [ ]:
import plotly.express as px


def plot_risk_return_scatter(results_df, max_points=1000):
    """

    Parameters
    ----------
    results_df
    max_points
    """
    df = results_df.dropna(
        subset=["expected_upside_pct", "expected_pt_hdi_lo", "expected_pt_hdi_hi", "last_price", "market_cap",
                "enterprise_value"]
    ).copy()
    # `results` carries expected_upside_pct (percent); convert to a decimal fraction.
    df["expected_upside"] = df["expected_upside_pct"] / 100.0
    # HDI band width as a fraction of last price = posterior uncertainty proxy
    df["uncertainty"] = (df["expected_pt_hdi_hi"] - df["expected_pt_hdi_lo"]) / df["last_price"]
    df["label"] = df["ticker"].fillna(df["isin"].str[:6])
    if len(df) > max_points:
        df = df.nlargest(max_points, "expected_upside")

    fig = px.scatter(
        df,
        x="uncertainty",
        y="expected_upside",
        color="sector",
        size="market_cap",
        size_max=22,
        hover_name="label",
        hover_data={"isin": True, "name": True, "industry": True, "market_cap": ":.2f", "enterprise_value": ":.2f",
                    "observed_pt": ":.2f", "expected_pt": ":.2f", "risk_adj_return": ":.2f",
                    "last_price": ":.2f", "uncertainty": ":.2%", "sector": True},
        title="Expected upside vs posterior uncertainty (94% HDI width)",
        labels={"uncertainty": "Posterior uncertainty (HDI width / last price)",
                "expected_upside": "Expected upside"},
    )
    fig.add_hline(y=0.0, line_dash="dot", line_color="grey")
    fig.update_layout(template="plotly_dark", height=650, legend_title_text="Sector")
    fig.show()


plot_risk_return_scatter(results)

### §14.3 — Posterior forest of top candidates (arviz)

In [ ]:
import matplotlib.pyplot as plt


def plot_top_candidate_forest(screen_ctx, results_df, top_n=20):
    """

    Parameters
    ----------
    screen_ctx
    results_df
    top_n

    Returns
    -------

    """
    # Rank by expected upside; drop names with no posterior screen value.
    top = (results_df.dropna(subset=["expected_upside_pct"])
           .sort_values("expected_upside_pct", ascending=False)
           .head(top_n))

    # screen.eu is (chain, draw, isin). Intersect with the posterior coord (preserving
    # the ranked order) so a name absent from the draws can't raise a sel KeyError.
    eu_isins = set(np.asarray(screen_ctx.eu["isin"].values).astype(str))
    top = top[top["isin"].astype(str).isin(eu_isins)]
    top_isins = top["isin"].astype(str).tolist()
    if not top_isins:
        print("No top candidates overlap the posterior draws — nothing to plot.")
        return
    eu = screen_ctx.eu.sel(isin=top_isins)

    # Ticker label with an ISIN-prefix fallback, built in ranked (top_isins) order.
    labels = [tk if isinstance(tk, str) and tk else str(iso)[:6]
              for iso, tk in zip(top["isin"], top["ticker"])]
    eu = eu.assign_coords(isin=labels)

    # arviz-plots 1.x expects a DataTree with a `posterior` group. Build it directly
    # from the existing DataArray rather than via azb.from_dict, which treats dict
    # *values* as nested {var: array} groups (a bare DataArray -> DataArray.items()).
    posterior = eu.rename("expected_upside").to_dataset(name="expected_upside")
    dt = DataTree.from_dict({"posterior": posterior})

    # 1.x renames hdi_prob -> ci_probs and *requires a two-element (inner, outer)
    # sequence*: plot_forest always indexes ci_probs[1], so a single-element list
    # raised IndexError. (0.5, 0.94) draws the inner 50% / outer 94% HDI. `labels`
    # and `figure_kwargs` mirror the module's canonical plot_forest call sites.
    figsize = (9.0, max(2.5, 0.42 * len(top_isins) + 1.0))
    pc = azp.plot_forest(
        dt,
        combined=True,
        ci_probs=(0.5, 0.94),
        labels=["isin"],
        backend="matplotlib",
        figure_kwargs={"figsize": figsize},
    )

    # 1.x returns a PlotCollection whose `plot` viz carries a `column` facet dim
    # (labels gutter + forest panel). Annotate only the forest column so the
    # break-even line isn't painted across the label gutter.
    forest_axes = pc.viz["plot"]
    if "column" in forest_axes.dims:
        forest_axes = forest_axes.sel(column="forest")
    for ax in np.atleast_1d(forest_axes.values).ravel():
        ax.axvline(0.0, color="#C44E52", ls="--", lw=1.5, zorder=0)

    pc.add_title(
        f"Posterior expected-upside forest — top {len(top_isins)} candidates "
        f"(inner 50% / outer 94% HDI)"
    )
    pc.show()


plot_top_candidate_forest(screen, results)


---
### Artifacts available at the top level

`kalman_df`, `roles`, `drift_features`, `panel`, `model`, `prior_idata`, `idata`,
`screen`, `results`, `kalman_results`, `single_ctx`, `mingled_ctx`, `forest_ctx`.

To run everything in one shot instead, call `kf.main(run_eda_section=True,
write_analytics=False, robust=True)`.